In [22]:
import pandas as pd
import requests
import json                 
from pprint import pprint

df = pd.DataFrame({
    "employee": ["emma", "noah", "sophia", "liam", "olivia", "william", "ava", "james", "isabella", "benjamin"],

    "adresse": [
        "10 rue de Rivoli, 75001 Paris",
        "5 avenue du Prado, 13008 Marseille",
        "12 place Bellecour, 69002 Lyon",
        "20 allée Jean Jaurès, 31000 Toulouse",
        "15 promenade des Anglais, 06000 Nice",
        "8 rue de la Fosse, 44000 Nantes",
        "25 rue des Grandes Arcades, 67000 Strasbourg",
        "18 boulevard Gambetta, 34000 Montpellier",
        "7 cours de l’Intendance, 33000 Bordeaux",
        "3 place du Théâtre, 59000 Lille"
    ]
})

for col in df.columns :
    df[col] = df[col].str.title()

In [23]:
print(df.head().to_markdown())

|    | employee   | adresse                              |
|---:|:-----------|:-------------------------------------|
|  0 | Emma       | 10 Rue De Rivoli, 75001 Paris        |
|  1 | Noah       | 5 Avenue Du Prado, 13008 Marseille   |
|  2 | Sophia     | 12 Place Bellecour, 69002 Lyon       |
|  3 | Liam       | 20 Allée Jean Jaurès, 31000 Toulouse |
|  4 | Olivia     | 15 Promenade Des Anglais, 06000 Nice |


Vous disposez d’un jeu de données contenant des informations sur des employés, notamment leur nom et leur adresse.

\
L’objectif de cet exercice est de visualiser la répartition géographique des employés sur une carte.

\
Pour cela, vous devrez :

+ transformer les adresses en coordonnées géographiques (latitude et longitude),

+ puis utiliser ces coordonnées pour les afficher sur une carte.

\
Afin d’atteindre cet objectif, vous devrez suivre plusieurs étapes successives, qui vous guideront de la préparation des données jusqu’à la visualisation finale.

\
👉 Cet exercice vise à vous entraîner à la préparation de données, à l’utilisation d’une API de géocodage, et à la visualisation cartographique.

In [24]:
# J'affiche mon url pour l'adresse demandée
url = "https://api-adresse.data.gouv.fr/search/?q=10+rue+de+Rivoli+Paris&postcode=75001"

# J'effectue ma requete grace a .get et .json pour l'avoir sous format json
result = requests.get(url)
pprint(result)

# JE VERIFIE LE STATUT DE MA REQUETE (se référer aux différents statut code)
print(f"Le code réponse est : {result.status_code} !")

# Je convertis en json et je l'affiche 
result.json()

<Response [200]>
Le code réponse est : 200 !


{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'geometry': {'type': 'Point', 'coordinates': [2.335861, 48.862527]},
   'properties': {'label': 'Rue de Rivoli 75001 Paris',
    'score': 0.7626387012987011,
    'id': '75101_8249',
    'name': 'Rue de Rivoli',
    'postcode': '75001',
    'citycode': '75101',
    'x': 651275.75,
    'y': 6862704.25,
    'city': 'Paris',
    'district': 'Paris 1er Arrondissement',
    'context': '75, Paris, Île-de-France',
    'type': 'street',
    'importance': 0.67474,
    'depcode': '75',
    'street': 'Rue de Rivoli',
    '_type': 'address'}}],
 'query': '10 rue de Rivoli Paris'}

Nous allons utiliser l'api adresse data gouv :

'https://data.geopf.fr/geocodage/search'

1. Explorer le chemin d'api permettant de recupérer la longitude et latitude à partir de cette adresse:

10 Rue De Rivoli, 75001 Paris

In [25]:
# je crée une fonction qui sort les coordonnées des adresse API

def API_address(postal_address):

  url = "https://api-adresse.data.gouv.fr/search/?q=" + postal_address.replace(" ","+")
  result = requests.get(url).json()
  coord = result['features'][0]['geometry']['coordinates'][::-1]
  return coord

In [26]:
# J'utilise ma fonction pour récuperer les coordonnées

API_address("10 Rue de Rivoli 75001 Paris")

[48.862527, 2.335861]

2. Appliquer le dernier travail à tous les employés , stocker la longitude et latitude de chaque employé dans dataframe.

In [27]:
# J'utilise apply pour appliquer ma fonction a tous les employés du dataframe en créant une nouvelle colonne 

df['coordonnees'] = df['adresse'].apply(API_address)

In [28]:
# J'affiche mon df avec la nouvelle colonne coordonnées
df.head()

,employee,adresse,coordonnees
0,Emma,"10 Rue De Rivoli, 75001 Paris","[48.862527, 2.335861]"
1,Noah,"5 Avenue Du Prado, 13008 Marseille","[43.272576, 5.386168]"
2,Sophia,"12 Place Bellecour, 69002 Lyon","[45.757597, 4.831662]"
3,Liam,"20 Allée Jean Jaurès, 31000 Toulouse","[43.605767, 1.449297]"
4,Olivia,"15 Promenade Des Anglais, 06000 Nice","[43.694941, 7.263031]"


3. Afficher tous les employé dans une map avec folium

In [ ]:
import folium

# je créer ma carte avec un zoom prédéfinit sur l'employee de Lyon ca centrera mieux ma carte, je choisis un zoom de 6 pour voir la France entièrement
m = folium.Map(location=df['coordonnees'][2], zoom_start=6)

# j'ajoute tous les marqueurs sur la même carte avec ma boucle
for i in range(len(df)):
    coord = df['coordonnees'][i]
    folium.Marker(
        location=coord,
        popup= df['employee'][i]
    ).add_to(m)

m  # j'affiche la carte

# Autre exercice

In [30]:
# Exercice 1 : nous avons la coordonnée d'un longitude et latitude d'un emplacement à paris et nous voulons connaitre l'adresse (avec API data gouv).
# 2.257903,48.837864

In [32]:
# je crée une fonction qui sort des adresses à partir des coordonnées

def API_reverse(coord):
    url = "https://api-adresse.data.gouv.fr/reverse/?lat=" + str(coord[0]) + "&lon=" + str(coord[1])
    result = requests.get(url).json()
    adresse = result['features'][0]['properties']['label']
    return adresse

In [35]:
# J'utilise ma fonction en entrant les coordonées, je met la latitude en premier et la longitude en deuxième pour que le resultat soit bon
print(f"L'adresse demandée est : {API_reverse([48.837864, 2.257903])}")

L'adresse demandée est : Place de la Porte de Saint-Cloud 75016 Paris,Paris 16e Arrondissement


In [31]:
# Exercice 2: Adapter vos codes pour refaire le meme travail avec l'api nominatim

https://nominatim.org/release-docs/latest/api/Overview/

In [ ]:
# Je crée une autre fonction pour nominatim, j'ajoute des headers, puis je suis la même logique, j'utilise ["display_name"] car c'est la clé qui correspond a l'adresse dans le json

def API_reverse2(coord):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:91.0) Gecko/20100101 Firefox/91.0',
        'Referer': 'https://www.example.com'
    }
    link = "https://nominatim.openstreetmap.org/reverse?lat=" + str(coord[0]) + "&lon=" + str(coord[1]) + "&format=json"
    result = requests.get(link, headers=headers).json()
    adresse = result['display_name']
    return adresse

In [48]:
# J'utilise ma fonction en entrant les coordonées avec Nominatim, je met la latitude en premier et la longitude en deuxième pour que le resultat soit bon
print(f"L'adresse demandée est : {API_reverse2([48.837864, 2.257903])}")

L'adresse demandée est : 4, Place de la Porte de Saint-Cloud, Quartier d'Auteuil, Paris 16e Arrondissement, Paris, Île-de-France, France métropolitaine, 75016, France


Questions :
+ Différence entre geocoding et reverse geocoding

+ C’est quoi une API

+ C’est quoi une API REST

+ Que signifie une réponse d’API de 200

+ Que signifie une réponse d’API de 403

+ Est-ce que c’est possible d’avoir une limite de requêtage d’une API

+ Sous quel format je récupère le retour d’une API

+ Quelles sont les opérations qui doivent être assurées par une API

- Le geocoding permet de recuperer les coordonnées et le reverse geocoding permet de recuperer une adresse à partir de coordonnées

- API veut dire "Application Programming Interface", une API est une interface qui permet à deux applications de communiquer. On envoie une requête selon des règles définies, et on reçoit une réponse, souvent en JSON.

- Une API REST doit suivre un ensemble de contraintes REST. Stateless (chaque requête est indépendante), utilisation des méthodes HTTP (GET, POST…), ressources identifiées par URL, format standardisé (JSON généralement).

- Une réponse 200 signifie un succès

- Une réponse 403 signifie une erreur. La requête est comprise mais le serveur refuse de la traiter. L'accès refusé, souvent par manque de permissions ou clé API invalide.

- Oui, il y a bien une limite de requêtage d'API, au bout d'un certain nombre, on peut etre bloqué et on devra le refaire le lendemain. On peut utiliser time.sleep aussi pour faire des requêtes moins agressives

- Sous format .json le plus souvent mais une API peut aussi retourner du XML, du CSV, du texte brut, ou du HTML.

- Les opérations sont POST, GET, PUT, DELETE. Il y a une mise en correspondance avec le CRUD (Create, Read, Update, Delete)